# 09. 8월 말 급등 보정 + 전주 라벨 역추론 최종판
08 노트북의 함수를 가져와 보정판 제출 파일만 빠르게 만든다 (08 전체 재실행은 40분 이상).
- 08 6.4절 결과: 08-25 공식 라벨 기준 (t̂, ŝ) = (1.25, 1.0) 범위 끝 → 범위 확장해 재추정
- 9/1 피처: 최근성 피처는 급등 보정값, `log_past_med`·`past_l`은 원값(공식 라벨이 원값 기준임을 08-11로 확인)
- 역추론 피처: 보정된 K와 t_eff = t̂ − ŝ 사용 (학습 조건과 같은 단위)
- 조건부 시뮬레이션: K 원값, U·V = 보정 분포 + ŝ, 임계값은 시뮬 풀의 80% 분위수

In [1]:
import os, glob, platform, warnings
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = {'Windows': 'Malgun Gothic', 'Darwin': 'AppleGothic'}.get(platform.system(), 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 50)

RAW = '../data/raw/epoch_data'
D = '2026-09-01'          # 본선 기준일
FUT = 26                  # 본선 정답 구간 길이 (일)
SEED = 42

con = duckdb.connect()
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    name = os.path.splitext(os.path.basename(p))[0]
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")

# # con.sql("SELECT MIN(published_at), MAX(published_at), COUNT(*) FROM video_5d_views").df()

In [2]:
def make_features(Dref):
    f = con.sql(f"""
    WITH lv AS (
        SELECT channel_id, view_5d, like_5d, comment_5d,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
    ),
    v5 AS (
        SELECT channel_id,
               COUNT(*)                                        AS n_long_5d,
               MEDIAN(view_5d)                                 AS past_med,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 7)     AS med_7,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 14)    AS med_14,
               MEDIAN(view_5d) FILTER (WHERE days_ago >= 14)   AS med_old,
               MEDIAN(like_5d / NULLIF(view_5d, 0))            AS like_rate,
               MEDIAN(comment_5d / NULLIF(view_5d, 0))         AS comment_rate,
               STDDEV(LN(1 + view_5d))                         AS log_view_std,
               QUANTILE_CONT(LN(1 + view_5d), 0.1)             AS log_q10,
               QUANTILE_CONT(LN(1 + view_5d), 0.9)             AS log_q90,
               REGR_SLOPE(LN(1 + view_5d), -days_ago)          AS trend_slope,
               MIN(days_ago)                                   AS days_since_meas
        FROM lv GROUP BY 1
    ),
    sv AS (
        SELECT channel_id, MEDIAN(view_5d) AS short_med
        FROM video_5d_views
        WHERE is_shorts = 1 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
        GROUP BY 1
    ),
    up AS (
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS n_long_7,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 28 DAY) AS n_long_28,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               AVG(duration_sec) FILTER (WHERE is_shorts = 0)                                                  AS avg_dur,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0  AS days_since_long
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1
    ),
    games AS (
        SELECT v.channel_id, COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}' GROUP BY 1
    )
    SELECT * FROM v5 LEFT JOIN sv USING (channel_id) LEFT JOIN up USING (channel_id)
                     LEFT JOIN games USING (channel_id)
    """).df()

    lp = np.log1p(f.past_med)
    f['log_past_med']      = lp
    f['mom_7']             = np.log1p(f.med_7) - lp
    f['mom_14']            = np.log1p(f.med_14) - lp
    f['mom_old']           = np.log1p(f.med_14) - np.log1p(f.med_old)
    f['short_ratio']       = f.n_short_14 / (f.n_short_14 + f.n_long_14).replace(0, np.nan)
    f['log_short_med_rel'] = np.log1p(f.short_med) - lp
    f['has_game']          = (f.n_games > 0).astype(int)
    f['log_n_long_5d']     = np.log1p(f.n_long_5d)
    return f

FEATURES = ['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long',
            'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate',
            'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur',
            'log_short_med_rel', 'log_n_long_5d']
print(len(FEATURES), 'features')
make_features(D)[['channel_id'] + FEATURES].head()

23 features


,channel_id,log_past_med,n_long_5d,like_rate,n_long_14,n_short_14,short_ratio,days_since_long,mom_14,log_view_std,n_games,has_game,mom_7,mom_old,comment_rate,log_q10,log_q90,trend_slope,days_since_meas,n_long_7,n_long_28,avg_dur,log_short_med_rel,log_n_long_5d
0,UCSzHok6X5qXEO7cjvVnE62g,11.041160,115,0.005344,33,3,0.083333,0.583333,0.551094,0.805554,0,0,0.576594,0.644188,0.000637,10.485301,12.464242,0.012643,0.583333,16,62,1461.066176,1.060906,4.753590
1,UC0tmAwhgk_1ATJtFyVHrFxg,11.042458,9,0.027933,5,0,0.000000,0.333333,0.051370,1.100363,0,0,0.291825,0.051370,0.004018,9.877976,11.702881,0.006593,0.333333,3,7,2254.363636,NaN,2.302585
2,UC24GgPBq5W3v34z0p0uk1Ow,7.052721,33,0.006061,11,0,0.000000,0.583333,0.453871,1.253764,1,1,1.436895,0.808324,0.000000,5.893977,8.898344,0.012117,0.583333,6,24,8978.926829,NaN,3.526361
3,UCL2ZLyAV0H9Pvi46W7TGdHQ,10.578954,36,0.006222,9,5,0.357143,1.541667,0.532194,0.720743,0,0,0.694076,0.685836,0.001486,10.082393,11.494362,0.011372,1.541667,6,19,1292.744186,2.582395,3.610918
4,UCDnIK80iouL_rb_J6GybOlg,1.098612,33,0.000000,7,0,0.000000,0.250000,0.000000,0.570446,0,0,0.287682,0.000000,0.000000,0.693147,1.755295,0.003839,0.250000,5,17,2954.684211,NaN,3.526361


In [3]:
def logreg(C=0.1):
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler()),
                     ('m', LogisticRegression(max_iter=2000, C=C, class_weight='balanced'))])

def rank01(x):
    x = pd.Series(np.asarray(x, dtype=float))
    return rankdata(x.fillna(x.median())) / len(x)

def blend(proba, df):
    """모델 확률 순위 + 과거 중앙값 역순위의 평균 (PR-AUC는 순위만 봄)"""
    return (rank01(proba) + rank01(-df.log_past_med)) / 2



In [4]:
def recent_feats(Dref):
    return con.sql(f"""
    WITH lv AS (
        SELECT channel_id, LN(1 + view_5d) AS l,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago,
               ROW_NUMBER() OVER (PARTITION BY channel_id ORDER BY published_at DESC) AS rn
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
    ), med AS (SELECT channel_id, MEDIAN(l) AS m FROM lv GROUP BY 1)
    SELECT lv.channel_id,
           SUM(l * EXP(-days_ago / 3.0))  / SUM(EXP(-days_ago / 3.0))  AS ew3,
           SUM(l * EXP(-days_ago / 7.0))  / SUM(EXP(-days_ago / 7.0))  AS ew7,
           SUM(l * EXP(-days_ago / 14.0)) / SUM(EXP(-days_ago / 14.0)) AS ew14,
           MAX(l) FILTER (WHERE rn = 1) AS last1,
           MEDIAN(l) FILTER (WHERE rn <= 3) AS last3,
           AVG(CASE WHEN l > med.m THEN 1.0 ELSE 0.0 END) FILTER (WHERE rn <= 5) AS hot5
    FROM lv JOIN med USING (channel_id) GROUP BY 1
    """).df()



In [5]:
from datetime import datetime, timedelta
shift = lambda d, k: (datetime.strptime(d, '%Y-%m-%d') + timedelta(days=k)).strftime('%Y-%m-%d')

def labels_full(Dref, fut):
    g = con.sql(f"""
        WITH agg AS (
          SELECT channel_id,
                 MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS past_score,
                 COUNT(view_5d)  FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS n_past,
                 COUNT(view_5d)  FILTER (WHERE published_at >= TIMESTAMP '{Dref}'
                                           AND published_at < TIMESTAMP '{Dref}' + INTERVAL {fut} DAY) AS n_fut,
                 COALESCE(MEDIAN(view_5d) FILTER (WHERE published_at >= TIMESTAMP '{Dref}'
                                           AND published_at < TIMESTAMP '{Dref}' + INTERVAL {fut} DAY), 0) AS future_score
          FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL GROUP BY 1)
        SELECT channel_id, LN(1 + past_score) AS past_l, n_fut,
               LN(1 + future_score) - LN(1 + past_score) AS growth
        FROM agg WHERE n_past >= 3 AND past_score >= 100""").df()
    t = np.quantile(g.growth, 0.8)
    g['target'] = (g.growth >= t).astype(int)
    g['ref_date'] = Dref
    return g, t

def known_window(D0, D1):
    """[D0, D1) 롱폼 view_5d (기지 구간 K) 를 채널별 리스트로"""
    k = con.sql(f"""SELECT channel_id, LIST(LN(1 + view_5d)) AS K
                    FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL
                      AND published_at >= TIMESTAMP '{D0}' AND published_at < TIMESTAMP '{D1}' GROUP BY 1""").df()
    return k

def past_level(Dref):
    return con.sql(f"""SELECT channel_id, LN(1 + MEDIAN(view_5d)) AS past_l_prev
                       FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL
                         AND published_at < TIMESTAMP '{Dref}' GROUP BY 1""").df()

PAIR_DATES = pd.date_range('2026-07-26', '2026-08-05', freq='D').strftime('%Y-%m-%d').tolist()
pieces, tinfo = [], []
for D_ in PAIR_DATES:
    cur, t_cur = labels_full(D_, 26)
    prv, t_prv = labels_full(shift(D_, -7), 18)
    tinfo.append((D_, round(t_cur, 4), round(t_prv, 4)))
    df = cur[['channel_id', 'ref_date', 'target', 'past_l']].merge(
        prv[['channel_id', 'target']].rename(columns={'target': 'prev'}), on='channel_id', how='left')
    df = (df.merge(past_level(shift(D_, -7)), on='channel_id', how='left')
            .merge(known_window(shift(D_, -7), D_), on='channel_id', how='left')
            .merge(make_features(D_), on='channel_id', how='left')
            .merge(recent_feats(D_), on='channel_id', how='left'))
    df['t_prev_true'] = t_prv
    pieces.append(df)
pairs = pd.concat(pieces, ignore_index=True)
pairs['ew3_d'] = pairs.ew3 - pairs.log_past_med
tinfo = pd.DataFrame(tinfo, columns=['D', 't_cur(FUT26)', 't_prev(D-7, FUT18)'])
print(tinfo.to_string())
print('t_cur 평균/표준편차', round(tinfo.iloc[:, 1].mean(), 4), round(tinfo.iloc[:, 1].std(), 4),
      '| t_prev 평균/표준편차', round(tinfo.iloc[:, 2].mean(), 4), round(tinfo.iloc[:, 2].std(), 4))
print('재현쌍', len(pairs), '행 | 전주 라벨 있는 비율', round(pairs.prev.notna().mean(), 3))
print('전주 라벨별 Rising 비율:', pairs.groupby(pairs.prev.fillna(-1)).target.mean().round(3).to_dict())

             D  t_cur(FUT26)  t_prev(D-7, FUT18)
0   2026-07-26        0.2650              0.2645
1   2026-07-27        0.2445              0.2558
2   2026-07-28        0.2837              0.2268
3   2026-07-29        0.2905              0.2343
4   2026-07-30        0.2670              0.2766
5   2026-07-31        0.2700              0.2752
6   2026-08-01        0.2727              0.2823
7   2026-08-02        0.3105              0.3042
8   2026-08-03        0.3874              0.2517
9   2026-08-04        0.4237              0.2737
10  2026-08-05        0.4600              0.2515
t_cur 평균/표준편차 0.3159 0.073 | t_prev 평균/표준편차 0.2633 0.0223
재현쌍 5102 행 | 전주 라벨 있는 비율 0.569
전주 라벨별 Rising 비율: {-1.0: 0.217, 0.0: 0.109, 1.0: 0.503}


In [6]:
T_PREV = float(tinfo.iloc[:, 2].mean())
T_CUR  = float(tinfo.iloc[:8, 1].mean())   # 08-03 이후는 t가 튀어 앞 8일 평균 사용
print('T_PREV', round(T_PREV, 4), '| T_CUR', round(T_CUR, 4))

def inv_feats(df, t_prev=T_PREV):
    out = pd.DataFrame(index=df.index)
    K = df.K.apply(lambda x: np.asarray(x, float) if isinstance(x, (list, np.ndarray)) else np.array([]))
    kn = K.apply(len); kmed = K.apply(lambda a: np.median(a) if len(a) else np.nan)
    c = df.past_l_prev + t_prev
    margin = kmed - c
    prev = df.prev
    out['prev_f'] = prev.fillna(0.2)
    out['prev_avail'] = prev.notna().astype(int)
    out['K_n'] = np.log1p(kn)
    out['margin'] = margin.fillna(-1.0)
    out['pos_low'] = (prev == 1) * np.clip(-margin.fillna(-1.0), 0, None)
    out['neg_high'] = (prev == 0) * np.clip(margin.fillna(-1.0), 0, None)
    out['prev_margin'] = out.prev_f * out.margin
    out['K_med_d'] = (kmed - df.log_past_med).fillna(0)
    return out
INV = ['prev_f', 'prev_avail', 'K_n', 'margin', 'pos_low', 'neg_high', 'prev_margin', 'K_med_d']

def chan_dist(Dref, days=28):
    return con.sql(f"""SELECT channel_id,
                              LIST(LN(1 + view_5d)) FILTER (WHERE published_at >= TIMESTAMP '{Dref}' - INTERVAL {days} DAY) AS Lr,
                              LIST(LN(1 + view_5d)) AS La
                       FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL
                         AND published_at < TIMESTAMP '{Dref}' GROUP BY 1""").df()

def _nanmed_rows(a):
    m = np.nanmedian(np.where(np.isnan(a).all(axis=1, keepdims=True), 0.0, a), axis=1)
    return m

def cond_sim(df, dists, t_prev=T_PREV, t_cur=T_CUR, n=400, jitter=0.3, u_days=11, v_days=15, seed=0):
    rng = np.random.default_rng(seed)
    dd = df[['channel_id']].merge(dists, on='channel_id', how='left')
    sc_c, sc_p, upost = np.zeros(len(df)), np.zeros(len(df)), np.full(len(df), np.nan)
    for j, (r, Lr, La) in enumerate(zip(df.itertuples(index=False), dd.Lr, dd.La)):
        L = np.asarray(Lr if isinstance(Lr, (list, np.ndarray)) and len(Lr) >= 3 else (La if isinstance(La, (list, np.ndarray)) else []), float)
        if len(L) == 0:
            continue
        rate = (r.n_long_28 if pd.notna(r.n_long_28) else 0) / 28 + 0.5 / 28
        ku, kv = rng.poisson(rate * u_days, n), rng.poisson(rate * v_days, n)
        mu_, mv_ = max(ku.max(), 1), max(kv.max(), 1)
        U = rng.choice(L, (n, mu_)) + rng.normal(0, jitter, (n, mu_)); U[np.arange(mu_)[None, :] >= ku[:, None]] = np.nan
        V = rng.choice(L, (n, mv_)) + rng.normal(0, jitter, (n, mv_)); V[np.arange(mv_)[None, :] >= kv[:, None]] = np.nan
        fut = _nanmed_rows(np.hstack([U, V]))
        hit = (fut - r.past_l) >= t_cur
        sc_p[j] = hit.mean()
        w = np.ones(n, bool)
        if pd.notna(r.prev) and pd.notna(r.past_l_prev):
            K = np.asarray(r.K, float) if isinstance(r.K, (list, np.ndarray)) else np.array([])
            KU = np.hstack([np.tile(K, (n, 1)), U]) if len(K) else U
            gp = _nanmed_rows(KU) - r.past_l_prev
            w = (gp >= t_prev) == bool(r.prev)
            if w.sum() < 5:
                w = np.ones(n, bool)
        sc_c[j] = hit[w].mean()
        um = _nanmed_rows(U)[w]
        upost[j] = np.mean(um[~np.isnan(um)]) - r.past_l if len(um) else np.nan
    return sc_c, sc_p, upost



T_PREV 0.2633 | T_CUR 0.2755


In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
DROP = ['comment_rate', 'avg_dur', 'log_q90', 'like_rate', 'log_q10', 'trend_slope', 'has_game']
PRUNED = [f for f in FEATURES if f not in DROP]
SIMF = ['sim_cond', 'u_post']

def hgb(seed):
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.03, max_iter=300, min_samples_leaf=40,
                                          l2_regularization=1.0, random_state=seed)
def p_lr(trn, tst, cols, C=0.1):
    return logreg(C).fit(trn[cols], trn.target).predict_proba(tst[cols])[:, 1]
def p_hgb(trn, tst, cols, seeds=5):
    return np.mean([hgb(s).fit(trn[cols], trn.target).predict_proba(tst[cols])[:, 1] for s in range(seeds)], axis=0)

def run_cv(df, comp_fns, seed=0, k=5):
    ch = df.channel_id.values; u = np.unique(ch); rg = np.random.default_rng(seed); rg.shuffle(u)
    fo = {c: i % k for i, c in enumerate(u)}; fold = np.array([fo[c] for c in ch])
    out = {n: np.zeros(len(df)) for n in comp_fns}
    for f in range(k):
        trn, tst = df[fold != f], df[fold == f]
        for n, fn in comp_fns.items():
            out[n][fold == f] = fn(trn, tst)
    return pd.DataFrame(out, index=df.index)

def summ(df, sc):
    R = {}
    for c in sc.columns:
        a = [average_precision_score(g.target, sc.loc[g.index, c]) for _, g in df.groupby('ref_date')]
        b = [average_precision_score(g.target, sc.loc[g.index, c]) for _, g in df[df.prev.notna()].groupby('ref_date')]
        R[c] = {'전체': np.mean(a), '전주있음': np.mean(b)}
    return pd.DataFrame(R).T


FPIS = PRUNED + ['ew3_d'] + INV + SIMF


In [8]:
vv = con.sql("""SELECT channel_id, CAST(DATE_TRUNC('day', published_at) AS DATE) AS day, is_shorts, LN(1 + view_5d) AS l
                FROM video_5d_views WHERE view_5d IS NOT NULL""").df()
def day_effect(v, iters=8):
    v = v.copy(); v['dlt'] = 0.0
    for _ in range(iters):
        a = (v.l - v.dlt).groupby(v.channel_id).median()
        v['a'] = v.channel_id.map(a)
        dl = (v.l - v.a).groupby(v.day).median()
        dl = dl - dl[dl.index < pd.Timestamp('2026-08-20')].median()   # 08-20 이전 기준 0
        v['dlt'] = v.day.map(dl)
    return dl, v
dl_long, vl = day_effect(vv[vv.is_shorts == 0])


In [9]:
# ---- 1. δ(day) 적용 뷰: 08-24 이후 롱폼만 보정 ----
DELTA = dl_long[dl_long.index >= pd.Timestamp('2026-08-24')].clip(lower=0)
delta_df = pd.DataFrame({'day': pd.to_datetime(DELTA.index), 'dlt': DELTA.values})
con.register('delta_df', delta_df)
con.sql(f"CREATE OR REPLACE VIEW v5_raw AS SELECT * FROM read_csv('{RAW}/video_5d_views.csv', header=true, auto_detect=true)")
def use_adjusted(flag):
    if flag:
        con.sql("""CREATE OR REPLACE VIEW video_5d_views AS
                   SELECT r.* REPLACE (CASE WHEN r.is_shorts = 0 AND r.view_5d IS NOT NULL AND d.dlt IS NOT NULL
                                            THEN EXP(LN(1 + r.view_5d) - d.dlt) - 1 ELSE r.view_5d END AS view_5d)
                   FROM v5_raw r LEFT JOIN delta_df d ON CAST(DATE_TRUNC('day', r.published_at) AS DATE) = CAST(d.day AS DATE)""")
    else:
        con.sql("CREATE OR REPLACE VIEW video_5d_views AS SELECT * FROM v5_raw")
use_adjusted(False)
print(DELTA.round(3).to_dict())

# ---- 2·3. 일반화된 조건부 시뮬레이션 ----
def cond_sim2(df, dists, s_u=0.0, t_prev=T_PREV, n=400, jitter=0.3, u_days=11, v_days=15, seed=0):
    """dists: (보정된) 채널 분포. K는 원값. U,V는 보정 분포 + s_u. 임계값은 시뮬 growth 풀의 80% 분위수"""
    rng = np.random.default_rng(seed)
    dd = df[['channel_id']].merge(dists, on='channel_id', how='left')
    G, W, UP = [], [], []
    for r, Lr, La in zip(df.itertuples(index=False), dd.Lr, dd.La):
        L = np.asarray(Lr if isinstance(Lr, (list, np.ndarray)) and len(Lr) >= 3 else (La if isinstance(La, (list, np.ndarray)) else []), float)
        if len(L) == 0:
            G.append(None); W.append(None); UP.append(np.nan); continue
        rate = (r.n_long_28 if pd.notna(r.n_long_28) else 0) / 28 + 0.5 / 28
        ku, kv = rng.poisson(rate * u_days, n), rng.poisson(rate * v_days, n)
        mu_, mv_ = max(ku.max(), 1), max(kv.max(), 1)
        U = rng.choice(L, (n, mu_)) + rng.normal(0, jitter, (n, mu_)) + s_u; U[np.arange(mu_)[None, :] >= ku[:, None]] = np.nan
        V = rng.choice(L, (n, mv_)) + rng.normal(0, jitter, (n, mv_)) + s_u; V[np.arange(mv_)[None, :] >= kv[:, None]] = np.nan
        g = _nanmed_rows(np.hstack([U, V])) - r.past_l
        w = np.ones(n, bool)
        if pd.notna(r.prev) and pd.notna(r.past_l_prev):
            K = np.asarray(r.K, float) if isinstance(r.K, (list, np.ndarray)) else np.array([])
            KU = np.hstack([np.tile(K, (n, 1)), U]) if len(K) else U
            w = (_nanmed_rows(KU) - r.past_l_prev >= t_prev) == bool(r.prev)
            if w.sum() < 5: w = np.ones(n, bool)
        G.append(g); W.append(w)
        um = _nanmed_rows(U)[w]; UP.append(np.nanmean(um) - s_u - r.past_l if len(um) else np.nan)
    pool = np.concatenate([g[w][np.random.default_rng(1).integers(0, w.sum(), 50)] for g, w in zip(G, W) if g is not None])
    t_cur = np.quantile(pool, 0.8)
    sc = np.array([np.mean(g[w] >= t_cur) if g is not None else np.nan for g, w in zip(G, W)])
    return np.nan_to_num(sc, nan=np.nanmean(sc)), np.array(UP), t_cur

def mle_su(df, dists, grid=np.round(np.arange(-0.3, 1.31, 0.05), 2), t_prev=T_PREV, n=300, jitter=0.3, u_days=11, seed=0):
    rng = np.random.default_rng(seed)
    dd = df[['channel_id']].merge(dists, on='channel_id', how='left')
    base = []
    for r, Lr, La in zip(df.itertuples(index=False), dd.Lr, dd.La):
        if pd.isna(r.prev) or pd.isna(r.past_l_prev): continue
        L = np.asarray(Lr if isinstance(Lr, (list, np.ndarray)) and len(Lr) >= 3 else (La if isinstance(La, (list, np.ndarray)) else []), float)
        if len(L) == 0: continue
        rate = (r.n_long_28 if pd.notna(r.n_long_28) else 0) / 28 + 0.5 / 28
        ku = rng.poisson(rate * u_days, n); mu_ = max(ku.max(), 1)
        U = rng.choice(L, (n, mu_)) + rng.normal(0, jitter, (n, mu_)); U[np.arange(mu_)[None, :] >= ku[:, None]] = np.nan
        K = np.asarray(r.K, float) if isinstance(r.K, (list, np.ndarray)) else np.array([])
        base.append((int(r.prev), K, U, r.past_l_prev))
    ll = []
    for s in grid:
        tot = 0.0
        for y, K, U, pp in base:
            KU = np.hstack([np.tile(K, (U.shape[0], 1)), U + s]) if len(K) else U + s
            p = np.clip(((_nanmed_rows(KU) - pp) >= t_prev).mean(), 0.01, 0.99)
            tot += np.log(p if y == 1 else 1 - p)
        ll.append(tot)
    return float(grid[int(np.argmax(ll))]), pd.Series(ll, index=grid)



{Timestamp('2026-08-24 00:00:00'): 0.045, Timestamp('2026-08-25 00:00:00'): 0.092, Timestamp('2026-08-26 00:00:00'): 0.326, Timestamp('2026-08-27 00:00:00'): 0.661, Timestamp('2026-08-28 00:00:00'): 0.759, Timestamp('2026-08-29 00:00:00'): 0.804, Timestamp('2026-08-30 00:00:00'): 0.789, Timestamp('2026-08-31 00:00:00'): 0.782}


In [10]:
def mle_ts(df, dists, t_grid, s_grid, n=300, jitter=0.3, u_days=11, seed=0):
    rng = np.random.default_rng(seed)
    dd = df[['channel_id']].merge(dists, on='channel_id', how='left')
    base = []
    for r, Lr, La in zip(df.itertuples(index=False), dd.Lr, dd.La):
        if pd.isna(r.prev) or pd.isna(r.past_l_prev): continue
        L = np.asarray(Lr if isinstance(Lr, (list, np.ndarray)) and len(Lr) >= 3 else (La if isinstance(La, (list, np.ndarray)) else []), float)
        if len(L) == 0: continue
        rate = (r.n_long_28 if pd.notna(r.n_long_28) else 0) / 28 + 0.5 / 28
        ku = rng.poisson(rate * u_days, n); mu_ = max(ku.max(), 1)
        U = rng.choice(L, (n, mu_)) + rng.normal(0, jitter, (n, mu_)); U[np.arange(mu_)[None, :] >= ku[:, None]] = np.nan
        K = np.asarray(r.K, float) if isinstance(r.K, (list, np.ndarray)) else np.array([])
        base.append((int(r.prev), K, U, r.past_l_prev))
    LL = np.zeros((len(t_grid), len(s_grid)))
    for j, s in enumerate(s_grid):
        meds = [(_nanmed_rows(np.hstack([np.tile(K, (U.shape[0], 1)), U + s]) if len(K) else U + s) - pp, y) for y, K, U, pp in base]
        for i, t in enumerate(t_grid):
            tot = 0.0
            for g, y in meds:
                p = np.clip((g >= t).mean(), 0.01, 0.99); tot += np.log(p if y == 1 else 1 - p)
            LL[i, j] = tot
    i, j = np.unravel_index(np.argmax(LL), LL.shape)
    return float(t_grid[i]), float(s_grid[j]), pd.DataFrame(LL, index=t_grid, columns=s_grid)



## 1. 09-01 테스트 데이터 (원값 + 보정값) 및 (t, s) 재추정 (범위 확장)

In [11]:
sample = pd.read_csv(f'{RAW}/sample_submission.csv', encoding='utf-8-sig')
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
prev825 = tl[tl.row_id.str.endswith('2026-08-25')][['channel_id', 'target']].rename(columns={'target': 'prev'})

use_adjusted(False)
raw_f = make_features(D)
test = (sample[['row_id', 'channel_id']]
        .merge(past_level(D).rename(columns={'past_l_prev': 'past_l'}), on='channel_id', how='left')
        .merge(prev825, on='channel_id', how='left')
        .merge(past_level('2026-08-25'), on='channel_id', how='left')
        .merge(known_window('2026-08-25', D), on='channel_id', how='left'))          # K 원값
use_adjusted(True)
adj_f = make_features(D); adj_r = recent_feats(D)
K_adj = known_window('2026-08-25', D).rename(columns={'K': 'K_adj'})
dist_adj = chan_dist(D)
use_adjusted(False)
# 최근성 피처는 보정값, log_past_med 는 원값
adj_f['log_past_med'] = adj_f[['channel_id']].merge(raw_f[['channel_id', 'log_past_med']], on='channel_id', how='left').log_past_med.values
test = test.merge(adj_f, on='channel_id', how='left').merge(adj_r, on='channel_id', how='left').merge(K_adj, on='channel_id', how='left')
test['ew3_d'] = test.ew3 - test.log_past_med
print('행', len(test), '| 전주 라벨', test.prev.notna().sum())
print('mom_7 평균: 원값', round(raw_f.mom_7.mean(), 3), '→ 보정', round(adj_f.mom_7.mean(), 3),
      '| 학습쌍 평균', round(pairs.mom_7.mean(), 3))

TG2 = np.round(np.arange(0.6, 2.01, 0.05), 2)
SG2 = np.round(np.arange(0.3, 1.81, 0.1), 2)
T_HAT, S_HAT, LL2 = mle_ts(test, dist_adj, TG2, SG2)
print(f'(t̂, ŝ) = ({T_HAT}, {S_HAT}) | t̂−ŝ = {T_HAT - S_HAT:.3f}')
print('로그우도 상위:', LL2.stack().sort_values(ascending=False).head(6).round(1).to_dict())

행 694 | 전주 라벨 666
mom_7 평균: 원값 0.564 → 보정 0.055 | 학습쌍 평균 0.005


(t̂, ŝ) = (1.25, 1.0) | t̂−ŝ = 0.250
로그우도 상위: {(1.25, 1.0): -203.8, (1.2, 1.0): -206.3, (1.2, 0.9): -206.6, (1.25, 1.1): -207.0, (1.3, 1.1): -208.7, (1.15, 0.9): -209.8}


### 1.1 결과
- (t̂, ŝ) = (1.25, 1.0), 범위 안쪽 최댓값. **t̂−ŝ = 0.25** = 평소 임계값(0.26)과 일치 → 9월 영상도 8월 말과 같은 수준으로 부풀려져 있다는 해석이 일관됨.
- 9/1 mom_7 평균: 원값 0.564 vs 학습 0.005 → 기존 모델은 본선에서 전혀 다른 분포의 피처를 받고 있었음. 보정 후 0.055.

## 2. 학습쌍 시뮬레이션(cond_sim2, 풀 기반 임계값) + 빠른 CV 확인 + 최종 예측

In [12]:
# 학습쌍: s=0, t=T_PREV, 시뮬 임계값은 풀 기반
pairs = pairs.join(inv_feats(pairs))
parts = []
for D_ in PAIR_DATES:
    m = pairs.ref_date == D_
    sc, up, _ = cond_sim2(pairs[m], chan_dist(D_), s_u=0.0, t_prev=T_PREV)
    parts.append(pd.DataFrame({'sim_cond': sc, 'u_post': up}, index=pairs.index[m]))
pairs = pairs.join(pd.concat(parts))

res = []
for seed in [0, 1]:
    raw = run_cv(pairs, {'lrP': lambda a, b: p_lr(a, b, FPIS, C=0.01),
                         'hgbP': lambda a, b: p_hgb(a, b, FPIS, seeds=3),
                         'simc': lambda a, b: b.sim_cond.values}, seed=seed)
    R = lambda c: raw[c].groupby(pairs.ref_date).rank(pct=True)
    sc_ = pd.DataFrame({'M3 LR': R('lrP'), 'M4 HGB+LR': 0.5 * R('hgbP') + 0.5 * R('lrP'),
                        'M5 M4+시뮬0.3': 0.35 * R('hgbP') + 0.35 * R('lrP') + 0.3 * R('simc'), 'S 시뮬 단독': R('simc')}, index=pairs.index)
    res.append(summ(pairs, sc_))
print(pd.concat(res).groupby(level=0).mean().round(4))

                 전체    전주있음
M3 LR        0.5066  0.5616
M4 HGB+LR    0.4983  0.5589
M5 M4+시뮬0.3  0.4945  0.5577
S 시뮬 단독      0.4576  0.5156


In [13]:
# 테스트: 역추론 피처는 보정 K + t_eff = t̂ − ŝ (학습과 같은 단위)
tinv = test.drop(columns=['K']).rename(columns={'K_adj': 'K'})
test = test.join(inv_feats(tinv, t_prev=T_HAT - S_HAT))
# 조건부 시뮬: K 원값, U·V = 보정 분포 + ŝ, t = t̂
sc_t, up_t, tcur_t = cond_sim2(test, dist_adj, s_u=S_HAT, t_prev=T_HAT)
test['sim_cond'], test['u_post'] = sc_t, up_t
print('시뮬 임계값(풀 80%):', round(tcur_t, 3), '| 전주 라벨별 시뮬 평균:', test.groupby(test.prev.fillna(-1)).sim_cond.mean().round(3).to_dict())

lr_fin = logreg(0.01).fit(pairs[FPIS], pairs.target)
p_lrP = lr_fin.predict_proba(test[FPIS])[:, 1]
p_hgbP = np.mean([hgb(s).fit(pairs[FPIS], pairs.target).predict_proba(test[FPIS])[:, 1] for s in range(5)], axis=0)

def save_sub(score, fname):
    s = np.asarray(score, float) + 1e-6 * rank01(p_lrP)
    s = (s - s.min()) / (s.max() - s.min())
    sub = sample[['row_id']].merge(pd.DataFrame({'row_id': test.row_id, 'prediction': s}), on='row_id', how='left')
    assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == 694 and sub.row_id.is_unique
    assert set(sub.row_id) == set(sample.row_id)
    assert sub.prediction.notna().all() and np.isfinite(sub.prediction).all() and sub.prediction.between(0, 1).all()
    sub.to_csv('../submissions/' + fname, index=False, encoding='utf-8')
    return sub.set_index('row_id').prediction

s15 = save_sub(0.5 * rank01(p_hgbP) + 0.5 * rank01(p_lrP), 'submit_15_M4fix_shockadj_previnv.csv')
s16 = save_sub(0.35 * rank01(p_hgbP) + 0.35 * rank01(p_lrP) + 0.3 * rank01(test.sim_cond), 'submit_16_M5fix_shockadj_previnv_sim.csv')
for nm, f in [('submit_04 (LB 0.4554)', 'submit_04_logreg_rankblend_fut26.csv'), ('submit_13 (보정 전 M4)', 'submit_13_M4_previnv_hgb_lr_sim.csv')]:
    o_ = pd.read_csv('../submissions/' + f).set_index('row_id').prediction
    print(f'{nm} 과 순위상관: submit_15 {s15.corr(o_.loc[s15.index], method="spearman"):.3f} | submit_16 {s16.corr(o_.loc[s16.index], method="spearman"):.3f}')
# 8월 말 업로드가 많은 채널의 순위 변화 (보정 전 → 후)
late_n = con.sql("""SELECT channel_id, COUNT(*) n_late FROM v5_raw WHERE is_shorts = 0 AND view_5d IS NOT NULL
                    AND published_at >= TIMESTAMP '2026-08-26' AND published_at < TIMESTAMP '2026-09-01' GROUP BY 1""").df()
tt = test[['row_id', 'channel_id']].merge(late_n, on='channel_id', how='left').fillna({'n_late': 0})
s13 = pd.read_csv('../submissions/submit_13_M4_previnv_hgb_lr_sim.csv').set_index('row_id').prediction
tt['r13'] = rank01(s13.loc[tt.row_id].values); tt['r15'] = rank01(s15.loc[tt.row_id].values)
tt['grp'] = pd.cut(tt.n_late, [-1, 0, 2, 5, 1000], labels=['0편', '1-2편', '3-5편', '6편+'])
print('8월 말(08-26~31) 업로드 수별 평균 순위 (보정 전 submit_13 → 보정 후 submit_15):')
print(tt.groupby('grp', observed=True)[['r13', 'r15']].mean().round(3).assign(n=tt.groupby('grp', observed=True).size()))

시뮬 임계값(풀 80%): 1.152 | 전주 라벨별 시뮬 평균: {-1.0: 0.179, 0.0: 0.128, 1.0: 0.496}


submit_04 (LB 0.4554) 과 순위상관: submit_15 0.685 | submit_16 0.628
submit_13 (보정 전 M4) 과 순위상관: submit_15 0.862 | submit_16 0.827
8월 말(08-26~31) 업로드 수별 평균 순위 (보정 전 submit_13 → 보정 후 submit_15):
        r13    r15    n
grp                    
0편    0.631  0.665  115
1-2편  0.539  0.554  262
3-5편  0.426  0.424  177
6편+   0.417  0.363  140


## 3. 리더보드 결과 반영: 과거 기준선 디플레이트 버전 (submit_17)
- LB: submit_15 0.6110 / 팀원 10_s2 0.6150 / **팀원 10_s3(s2 + 부풀려진 과거 기준선 보정) 0.6290**
- 08-11 재현(원값 0.985 > 보정값 0.945)은 *미래 구간* 근거였고, 9/1 *과거 구간*은 직접 확인 불가 → LB로 검증.
- 변경점(나머지는 submit_15와 동일): `log_past_med`·`past_l` 보정값, 시뮬은 보정 세계(K 보정값, s=0, t = t̂−ŝ = 0.25)

In [14]:
use_adjusted(True)
past_adj = make_features(D)[['channel_id', 'log_past_med']].rename(columns={'log_past_med': 'lpm_adj'})
pl_adj = past_level(D).rename(columns={'past_l_prev': 'past_l_adj'})
use_adjusted(False)
t17 = test.merge(past_adj, on='channel_id', how='left').merge(pl_adj, on='channel_id', how='left')
print('과거 중앙값이 바뀐 채널 수:', int((t17.lpm_adj - t17.log_past_med).abs().gt(1e-9).sum()),
      '| 평균 변화', round((t17.lpm_adj - t17.log_past_med).mean(), 3))
t17['log_past_med'] = t17.lpm_adj
t17['past_l'] = t17.past_l_adj
t17['ew3_d'] = t17.ew3 - t17.log_past_med
inv17 = inv_feats(t17.drop(columns=['K']).rename(columns={'K_adj': 'K'}), t_prev=T_HAT - S_HAT)
for c in INV:
    t17[c] = inv17[c].values
sc17, up17, tc17 = cond_sim2(t17.drop(columns=['K']).rename(columns={'K_adj': 'K'}), dist_adj, s_u=0.0, t_prev=T_HAT - S_HAT)
t17['sim_cond'], t17['u_post'] = sc17, up17
print('시뮬 임계값:', round(tc17, 3))

p_lr17 = lr_fin.predict_proba(t17[FPIS])[:, 1]
p_hgb17 = np.mean([hgb(s).fit(pairs[FPIS], pairs.target).predict_proba(t17[FPIS])[:, 1] for s in range(5)], axis=0)
s = 0.5 * rank01(p_hgb17) + 0.5 * rank01(p_lr17) + 1e-6 * rank01(p_lr17)
s = (s - s.min()) / (s.max() - s.min())
sub17 = sample[['row_id']].merge(pd.DataFrame({'row_id': t17.row_id, 'prediction': s}), on='row_id', how='left')
assert list(sub17.columns) == ['row_id', 'prediction'] and len(sub17) == 694 and sub17.row_id.is_unique
assert set(sub17.row_id) == set(sample.row_id)
assert sub17.prediction.notna().all() and np.isfinite(sub17.prediction).all() and sub17.prediction.between(0, 1).all()
sub17.to_csv('../submissions/submit_17_M4fix_pastdeflate.csv', index=False, encoding='utf-8')
s17 = sub17.set_index('row_id').prediction
print('submit_15 와 순위상관:', round(s17.corr(s15.loc[s17.index], method='spearman'), 4))
tt['r17'] = rank01(s17.loc[tt.row_id].values)
print('8월 말 업로드 수별 평균 순위 (submit_15 → submit_17):')
print(tt.groupby('grp', observed=True)[['r15', 'r17']].mean().round(3))

과거 중앙값이 바뀐 채널 수: 426 | 평균 변화 -0.082


시뮬 임계값: 0.193


submit_15 와 순위상관: 0.9242
8월 말 업로드 수별 평균 순위 (submit_15 → submit_17):
        r15    r17
grp               
0편    0.665  0.634
1-2편  0.554  0.538
3-5편  0.424  0.435
6편+   0.363  0.404
